In [40]:
import numpy as np
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

In [41]:
import os
root_path = r"C:\gtsrb_data"
os.makedirs(root_path, exist_ok=True)

In [42]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])
train_dataset = datasets.GTSRB(root=root_path, split="train", download=True, transform=transform)
test_dataset  = datasets.GTSRB(root=root_path, split="test",  download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=len(test_dataset),  shuffle=False)
X_train_full, y_train_full = next(iter(train_loader))
X_test, y_test = next(iter(test_loader))

In [43]:
X_train_full = X_train_full.permute(0, 2, 3, 1).numpy()
y_train_full = y_train_full.numpy()
X_test = X_test.permute(0, 2, 3, 1).numpy()
y_test = y_test.numpy()
print(f"raw data train: {X_train_full.shape}, test: {X_test.shape}")


raw data train: (26640, 32, 32, 3), test: (12630, 32, 32, 3)


In [44]:
def clean_data(X, y):
    brightness = X.mean(axis=(1, 2, 3))
    mask = (brightness > 0.05) & (brightness < 0.95)

    variance = X.var(axis=(1, 2, 3))
    mask &= variance > 1e-4

    return X[mask], y[mask]

X_train_full, y_train_full = clean_data(X_train_full, y_train_full)
X_test, y_test = clean_data(X_test, y_test)

print(f"clean data  train: {X_train_full.shape}, test: {X_test.shape}")

clean data  train: (26384, 32, 32, 3), test: (12574, 32, 32, 3)


In [45]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15,
    stratify=y_train_full,
    random_state=42
)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Train: (22426, 32, 32, 3), Val: (3958, 32, 32, 3), Test: (12574, 32, 32, 3)


In [46]:
num_classes = 43
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat   = to_categorical(y_val, num_classes)
y_test_cat  = to_categorical(y_test, num_classes)

In [47]:
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.7, 1.3],
    zoom_range=0.1
    
)
datagen.fit(X_train)

In [54]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 43)             │        11,051 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 628,843 (2.40 MB)

 Trainable params: 628,843 (2.40 MB)

 Non-trainable params: 0 (0.00 B)

In [49]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True)
]

In [50]:
import os
save_path = r"C:\gtsrb_data\best_model.keras"
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
    ModelCheckpoint(filepath=save_path, monitor='val_accuracy', save_best_only=True)
]

In [55]:
history = model.fit(
    X_train, y_train_cat, batch_size=64,
    validation_data=(X_val, y_val_cat),
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 18s 40ms/step - accuracy: 0.3131 - loss: 2.4270 - val_accuracy: 0.6117 - val_loss: 1.2023 - learning_rate: 0.0010
Epoch 2/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 16s 45ms/step - accuracy: 0.6972 - loss: 0.9216 - val_accuracy: 0.9121 - val_loss: 0.3183 - learning_rate: 0.0010
Epoch 3/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 15s 43ms/step - accuracy: 0.8780 - loss: 0.3784 - val_accuracy: 0.9553 - val_loss: 0.1529 - learning_rate: 0.0010
Epoch 4/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 15s 43ms/step - accuracy: 0.9314 - loss: 0.2145 - val_accuracy: 0.9836 - val_loss: 0.0681 - learning_rate: 0.0010
Epoch 5/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 15s 43ms/step - accuracy: 0.9573 - loss: 0.1382 - val_accuracy: 0.9899 - val_loss: 0.0418 - learning_rate: 0.0010
Epoch 6/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 16s 45ms/step - accuracy: 0.9703 - loss: 0.0983 - val_accuracy: 0.9907 - val_loss: 0.0324 - learning_rate: 0.0010
Epoch 7/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 15s 44ms/step - accuracy: 0.9761 - l

In [56]:

test_loss, test_acc = model.evaluate(X_test, y_test_cat)
print(f"Test accuracy : {test_acc*100:.2f}%")

393/393 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9345 - loss: 0.3667
Test accuracy : 93.45%


In [ ]:
from tensorflow.keras.models import load_model
from PIL import Image
model = load_model(r"C:\gtsrb_data\best_model.keras")

def predict_image(image_path, img_size=32):
    img = Image.open(image_path).convert("RGB")
    img = img.resize((img_size, img_size))
    img_array = np.array(img).astype("float32") / 255.0
    
    img_array = np.expand_dims(img_array, axis=0)
    
    predictions = model.predict(img_array)
    predicted_class = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class]
    
    return predicted_class, confidence


In [68]:
CLASS_NAMES = {
    0: "Speed limit (20km/h)", 1: "Speed limit (30km/h)", 2: "Speed limit (50km/h)",
    3: "Speed limit (60km/h)", 4: "Speed limit (70km/h)", 5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)", 7: "Speed limit (100km/h)", 8: "Speed limit (120km/h)",
    9: "No passing", 10: "No passing (vehicles > 3.5t)", 11: "Right-of-way at intersection",
    12: "Priority road", 13: "Yield", 14: "Stop", 15: "No vehicles",
    16: "Vehicles > 3.5t prohibited", 17: "No entry", 18: "General caution",
    19: "Dangerous curve left", 20: "Dangerous curve right", 21: "Double curve",
    22: "Bumpy road", 23: "Slippery road", 24: "Road narrows on the right",
    25: "Road work", 26: "Traffic signals", 27: "Pedestrians", 28: "Children crossing",
    29: "Bicycles crossing", 30: "Beware of ice/snow", 31: "Wild animals crossing",
    32: "End speed + passing limits", 33: "Turn right ahead", 34: "Turn left ahead",
    35: "Ahead only", 36: "Go straight or right", 37: "Go straight or left",
    38: "Keep right", 39: "Keep left", 40: "Roundabout mandatory",
    41: "End of no passing", 42: "End no passing (vehicles > 3.5t)"
}


In [71]:
import matplotlib as plt
class_id, conf = predict_image("C:\\Users\\User\\Desktop\\ماشین خودران\\002.jpg")
print(f"sign: {CLASS_NAMES[class_id]} ({conf:.2%})")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
sign: Speed limit (60km/h) (99.99%)
